In [2]:
import os
import logging

import numpy as np
import pandas as pd
import healpy as hp

from matplotlib import pyplot as plt

from regressis.dataframe import PhotometricDataFrame
from regressis.systematics import plot_systematic_from_map
from regressis import Regression
from regressis.utils import setup_logging, setup_mplstyle, read_fits_to_pandas, build_healpix_map
from regressis.footprint import DR9Footprint, DESIFootprint
from regressis.plot import plot_moll

dict_moll_param = {'figsize':(3.5, 2.0), 'xpad':1.35, 'labelpad':-16.5, 'ycb_pos':-0.10, 'xlabel_labelpad':5.0}

In [ ]:
LSS = f'/global/u2/e/edmondc/desi_catalogs/DA2/LSS/loa-v1/LSScats/v1.1/'
randoms_fn, data_fn = '{}_{}_full.ran.fits', '{}_full.dat.fits'

In [ ]:
logger = logging.getLogger('MAIN')

# To avoid error from pandas method into the logger -> pandas use NUMEXPR Package
os.environ.setdefault('NUMEXPR_MAX_THREADS', os.environ.get('OMP_NUM_THREADS', '1'))
os.environ.setdefault('NUMEXPR_NUM_THREADS', os.environ.get('OMP_NUM_THREADS', '1'))

setup_logging()
setup_mplstyle()

In [ ]:
# Plot param:
cmap = plt.get_cmap('jet', 10).copy()
cmap.set_extremes(under='darkgrey')

In [ ]:
nbr_randoms = 4

tracer, zmin, zmax = 'QSO', 0.8, 3.1

In [ ]:
columns = ['TARGETID', 'RA', 'DEC', 'Z_not4clus', 'FRACZ_TILELOCID', 'FRAC_TLOBS_TILES']
data = read_fits_to_pandas(os.path.join(LSS, data_fn.format(tracer)), columns=columns)
data = data[(data['Z_not4clus'] > zmin) & (data['Z_not4clus'] < zmax)]

columns = ['TARGETID', 'RA', 'DEC']
randoms = pd.concat([read_fits_to_pandas(os.path.join(LSS, randoms_fn.format(tracer, i)), columns=columns) for i in range(nbr_randoms)]) 

In [ ]:
data['WEIGHT_COMP'] = 1 / data['FRACZ_TILELOCID'] / data['FRAC_TLOBS_TILES']
data['Z'] = data['Z_not4clus']

In [ ]:
# Define Footprint:
nside = 256

footprint = DESIFootprint(nside)
dr9_footprint = DR9Footprint(nside, mask_around_des=False, cut_desi=False)

# Compute healpix
map_data = build_healpix_map(nside, data['RA'].values, data['DEC'].values, in_deg2=False)
map_data_wcomp = build_healpix_map(nside, data['RA'].values, data['DEC'].values, weights=data['WEIGHT_COMP'].values, in_deg2=False)

map_randoms_now = build_healpix_map(nside, randoms['RA'].values, randoms['DEC'].values, in_deg2=True)

fracarea_now = map_randoms_now / (nbr_randoms * 2500)
fracarea_now[fracarea_now == 0] = np.NaN
# remove pixels with too small fracarea
sel = 1/fracarea_now > 5.0
fracarea_now[sel] = np.NaN
fracarea = fracarea_now

# attention pour fracarea il faut bien prendre fracarea_now (ok ? sure de chez sure ?)
def count2moll(mp, fracarea=fracarea_now, footprint=footprint):
    mp_tmp = mp.copy() / hp.nside2pixarea(hp.npix2nside(mp.size), degrees=True) / fracarea
        
    mp_tmp[footprint('y5') & np.isnan(mp_tmp)] = -1
    return mp_tmp

In [ ]:
plot_moll(fracarea, min=0, max=1, cmap=cmap, label=r'fracarea', **dict_moll_param)
plot_moll(count2moll(map_data), min=10, max=225, galactic_plane=True, cmap=cmap, label=r'[$\#$ deg$^{-2}$]', **dict_moll_param)
plot_moll(count2moll(map_data_wcomp), min=10, max=225, galactic_plane=True, cmap=cmap, label=r'[$\#$ deg$^{-2}$]', filename=f'fig/{tracer}-y1-with_comp-with_fracarea.png', **dict_moll_param)

In [ ]:
import healpy as hp
data['HPX'] = hp.ang2pix(256, data['RA'], data['DEC'], nest=True, lonlat=True)

In [ ]:
print(f'\nNbr QSO in North: {9.72 / hp.nside2pixarea(256, degrees=True):2.2f} per deg^2')
print(f'Nbr QSO in South: {9.81 / hp.nside2pixarea(256, degrees=True):2.2f} per deg^2')
print(f'Nbr QSO in DES: {9.99 / hp.nside2pixarea(256, degrees=True):2.2f} per deg^2\n')

In [ ]:
version, tracer, suffix_tracer, nside = 'DA2', tracer, '', 256
suffix_regressor = ''
dr9_footprint = DR9Footprint(nside, mask_around_des=False, cut_desi=False)

param = {'data_dir': '/global/homes/e/edmondc/Software/regressis/data/',
         'output_dir': '.',
         'use_median': False,
         'use_new_norm': False, 
         'regions': ['North', 'South_ngc', 'South_sgc', 'Des']}

feature_names = ['EBV', 'STARDENS', 'STREAM',
                 'PSFSIZE_G', 'PSFSIZE_R', 'PSFSIZE_Z',
                 'PSFDEPTH_G', 'PSFDEPTH_R', 'PSFDEPTH_Z', 'PSFDEPTH_W1', 'PSFDEPTH_W2']

# probleme avec _ngc dans les controles plots :(
plt.rcParams.update(plt.rcParamsDefault)

dataframe = PhotometricDataFrame(version, tracer, dr9_footprint, suffix_tracer, **param)
dataframe.set_features()
dataframe.set_targets(targets=map_data_wcomp, fracarea=fracarea)
dataframe.build(cut_fracarea=False)

setup_mplstyle()

In [ ]:
data_sorted = data.sort_values('Z')

nbins = 10
nbr_obj_bins =  int(data['Z'].size / nbins)
bins = np.copy(data_sorted['Z'][0::nbr_obj_bins].values)
bins[-1] += 1
wbin = np.digitize(data_sorted['Z'], bins, right=False)

print(bins)
print(np.unique(wbin, return_counts=True))

maps, legends = [], []
for i in range(1, nbins+1):
    sel = wbin == i
    maps.append(build_healpix_map(nside, data_sorted['RA'][sel].values, data_sorted['DEC'][sel].values, weights=data_sorted['WEIGHT_COMP'][sel].values, in_deg2=True))
    maps[-1] /= fracarea
    legends.append(f"${bins[i-1]:2.2f}<z<{bins[i]:2.2f}$" if i < nbins else f"${bins[i-1]:2.2f}<z$")

colors = plt.get_cmap('jet')(np.linspace(0.1, 0.9, nbins))

with np.errstate(divide='ignore', invalid='ignore'):
    plot_systematic_from_map(maps,legends,
                             fracarea, dataframe.footprint, dataframe.features, dataframe.pixels, colors=colors, 
                             feature_names=dataframe.features_toplot[:11], 
                             regions=['South_ngc'],
                             ax_lim=0.1, cut_fracarea=False, legend_title=True, hist_legend=False, n_bins=10,
                             show=True, save=False, savedir='.')